In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from pmdarima import auto_arima

In [52]:
# 1. Carga de datos
file_path = r"/Users/alejandrabenavidessanclemente/Desktop/ProyectoGradoGEB/datos_procesados.csv"
df = pd.read_csv(file_path, parse_dates=['EFFECTIVE_DATE'], index_col='EFFECTIVE_DATE')

In [54]:
# 2. Definir variable objetivo
target = 'ORIG_RAW_VOLUME'

# Filtrar solo las variables numéricas para las características
X = df.drop(columns=[target]).select_dtypes(include=[np.number])
y = df[target]

In [55]:
# 3. Eliminar Variables no necesarias con la Correlación
# Calcular la correlación con la variable objetivo
corr_with_target = X.corrwith(y)

# Definir umbrales para eliminar variables redundantes
upper_threshold = 0.98  # Correlación muy alta → redundante
lower_threshold = 0.05   # Correlación muy baja → no aporta información

# Encontrar las variables a eliminar
columns_to_drop = corr_with_target[(corr_with_target.abs() > upper_threshold) | (corr_with_target.abs() < lower_threshold)].index.tolist()

# No eliminar la variable objetivo
if target in columns_to_drop:
    columns_to_drop.remove(target)

# Eliminar las columnas redundantes
X = X.drop(columns=columns_to_drop)

print(f"Columnas eliminadas por correlación: {columns_to_drop}")
print(f"Columnas restantes: {X.columns}")

Columnas eliminadas por correlación: ['ORIG_TEMPERATURE', 'TEMPERATURE', 'RAW_VOLUME']
Columnas restantes: Index(['ORIG_STD_VOLUME', 'STD_VOLUME', 'PRESSURE', 'ORIG_PRESSURE'], dtype='object')


In [56]:
# 4. Normalizar los datos
# Normalizar las variables explicativas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# División en entrenamiento y prueba (80% - 20%)
train_size = int(len(df) * 0.8)
X_train, X_test = X_scaled[:train_size], X_scaled[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print("Escalado realizado correctamente.")

Escalado realizado correctamente.


In [57]:
# 5. Ajuste Manual de los Hiperparámetros
# Parámetros SARIMAX manuales (ajusta estos valores)
p, d, q = 1, 1, 1  # ARIMA
P, D, Q, S = 1, 1, 1, 24  # Estacionalidad (24 para datos horarias)

# ===========================
# 5. Entrenar el modelo SARIMAX con los hiperparámetros ajustados manualmente
# ===========================

print(f"Entrenando el modelo SARIMAX con ARIMA({p},{d},{q}) y estacionalidad ({P},{D},{Q},{S})...")

from statsmodels.tsa.statespace.sarimax import SARIMAX

# Ajuste del modelo SARIMAX
model = SARIMAX(y_train, exog=X_train, order=(p, d, q), seasonal_order=(P, D, Q, S))
result = model.fit(disp=False)

Entrenando el modelo SARIMAX con ARIMA(1,1,1) y estacionalidad (1,1,1,24)...


/Users/alejandrabenavidessanclemente/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/alejandrabenavidessanclemente/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it is not monotonic and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/alejandrabenavidessanclemente/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/alejandrabenavidessanclemente/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but 

KeyboardInterrupt: 